# 5. Тематика соревнований
Сопоставляем теги с темами (CV, NLP, табличные, временные ряды) и смотрим динамику по годам.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load
from src.topics import competition_themes

sns.set_theme(style="whitegrid")
comp = load("Competitions")
comp = comp[comp["HostSegmentTitle"].isin(["Featured", "Research", "Playground", "Getting Started"])].copy()
comp["year"] = comp["EnabledDate"].dt.year

comp_tags = load("CompetitionTags").merge(load("Tags"), left_on="TagId", right_on="Id")
comp["theme"] = comp["Id"].map(competition_themes(comp_tags)).fillna("other")
comp["theme"].value_counts()

Если в "other" попадает много соревнований — расширить ключевые слова в `src/topics.py`.

In [ ]:
comp.groupby(["year", "theme"]).size().unstack(fill_value=0).plot(
    kind="bar", stacked=True, figsize=(12, 5), title="Соревнования по темам и годам"
)
plt.ylabel("число соревнований")
plt.savefig("../reports/figures/05_themes_by_year.png", bbox_inches="tight")
plt.show()

## Призы и число команд по темам

In [ ]:
comp["prize_usd"] = comp["RewardQuantity"].where(comp["RewardType"] == "USD")
comp.groupby("theme").agg(
    n=("Id", "size"),
    median_teams=("TotalTeams", "median"),
    median_prize_usd=("prize_usd", "median"),
)

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=comp, x="theme", y="TotalTeams", showfliers=False)
plt.title("Число команд по темам (без выбросов)")
plt.show()